In [24]:
import pandas as pd

In [25]:
df=pd.read_csv("train.txt",sep=";",header=None,names=["Text","Emotions"])

In [26]:
df.head()

,Text,Emotions
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [27]:
df.isnull().sum()

Text        0
Emotions    0
dtype: int64

In [28]:
df=df.drop_duplicates()

In [29]:
unique_emotion=df["Emotions"].unique()

In [30]:
unique_emotion

<StringArray>
['sadness', 'anger', 'love', 'surprise', 'fear', 'joy']
Length: 6, dtype: str

In [31]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["Emotions"] = le.fit_transform(df["Emotions"])

# Text Processing

<h3>Convert to Lowercase</h3>

In [32]:
def toLowerCase(text:str):
        return text.lower()
df["Text"]=df["Text"].apply(toLowerCase)

# Remove URLs

In [33]:
import re

def remove_urls(text: str) -> str:
    pattern = r'https?://\S+|www\.\S+'
    return re.sub(pattern, '', text)

df["Text"]=df["Text"].apply(remove_urls)

# Remove Digits

In [34]:
def remove_digits(text: str):
    return re.sub(r'\d+', '', text)

df["Text"]=df["Text"].apply(remove_digits)

# Remove Emojies

In [35]:

def remove_emojis(text: str) -> str:
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F" 
        "\U0001F300-\U0001F5FF" 
        "\U0001F680-\U0001F6FF" 
        "\U0001F1E0-\U0001F1FF" 
        "\U00002700-\U000027BF" 
        "\U0001F900-\U0001F9FF" 
        "\U00002600-\U000026FF"
        "]+",
        flags=re.UNICODE
    )
    return emoji_pattern.sub('', text)

df["Text"]=df["Text"].apply(remove_emojis)

# Remove Punctuations

In [36]:
import string

def remove_punctuation(text: str) -> str:
    return text.translate(str.maketrans('', '', string.punctuation))

df["Text"]=df["Text"].apply(remove_punctuation)

In [37]:
import nltk
# nltk.download('punkt_tab')
# nltk.download('stopwords')
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
stop_words = set(stopwords.words("english"))

In [38]:
def remove_stop_word(text:str)->str:
        tokens_word=word_tokenize(text)
        filtered_word=[
                word for word in tokens_word
                if word.lower() not in stop_words
        ]
        return " ".join(filtered_word)

remove_stop_word('i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake')

'go feeling hopeless damned hopeful around someone cares awake'

<h3>Remove Stop words because they don't effect in NLP</h3>

In [39]:
df["cleaned_text"]=df["Text"].apply(remove_stop_word)
df.tail()

,Text,Emotions,cleaned_text
15995,i just had a very brief time in the beanbag an...,4,brief time beanbag said anna feel like beaten
15996,i am now turning and i feel pathetic that i am...,4,turning feel pathetic still waiting tables sub...
15997,i feel strong and good overall,2,feel strong good overall
15998,i feel like this was such a rude comment and i...,0,feel like rude comment im glad
15999,i know a lot but i feel so stupid because i ca...,4,know lot feel stupid portray


<h3>Using Bag of words instead of One-hot encoding to convert documents into vectors</h3>

In [40]:
# Ignore this code as i was testing how it works TESTING
from sklearn.feature_extraction.text import CountVectorizer
vect=CountVectorizer(ngram_range=(2,2))
documents = [
    "I like Python",
    "AI is useful",
    "Machine learning is interesting"
]
model=vect.fit_transform(documents)

In [41]:
print(vect.get_feature_names_out(documents))
print(model.toarray()) # type: ignore

['ai is' 'is interesting' 'is useful' 'learning is' 'like python'
 'machine learning']
[[0 0 0 0 1 0]
 [1 0 1 0 0 0]
 [0 1 0 1 0 1]]


In [42]:
from sklearn.metrics import recall_score,precision_score,accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
BOW=CountVectorizer(ngram_range=(1,2),stop_words="english")
TF_IDF=TfidfVectorizer(ngram_range=(1,2))
X_train, X_test, y_train, y_test = train_test_split(df["cleaned_text"], df["Emotions"], test_size=0.20, random_state=42)

<h3>Using Bag of words</h3>

In [50]:
X_train_BOW=BOW.fit_transform(X_train)
X_test_BOW=BOW.transform(X_test)

models={
        "Logistic Regression":LogisticRegression(max_iter=1000,solver= 'lbfgs', C= 10),
}

results=[]
for name,model in models.items():
        model.fit(X_train_BOW,y_train)
        y_predict=model.predict(X_test_BOW)
        results.append({
                "Model Name: ":name,
                "Accuracy: ":accuracy_score(y_test,y_predict),
                "Precision: ":precision_score(y_test,y_predict,average="weighted"),
                "Recall: ":recall_score(y_test,y_predict,average="weighted")
        })

In [52]:
pd.DataFrame(data=results)

,Model Name:,Accuracy:,Precision:,Recall:
0,Logistic Regression,0.907188,0.906827,0.907188


In [53]:
import joblib
joblib.dump(models["Logistic Regression"],"model.pkl")
joblib.dump(le,"LabelEncoder.pkl")
joblib.dump(BOW,"vectorizer.pkl")

['vectorizer.pkl']

<h3>Using TF-IDF</h3>

In [47]:
X_train_TF_IDF=TF_IDF.fit_transform(X_train)
X_test_TF_IDF=TF_IDF.transform(X_test)

models={
        "Logistic Regression":LogisticRegression(max_iter=1000,solver="lbfgs",C=10),
}

results=[]
for name,model in models.items():
        model.fit(X_train_TF_IDF,y_train)
        y_predict=model.predict(X_test_TF_IDF)
        results.append({
                "Model Name":name,
                "Accuracy":accuracy_score(y_test,y_predict),
                "Precision":precision_score(y_test,y_predict,average="weighted"),
                "Recall":recall_score(y_test,y_predict,average="weighted")
        })

In [48]:
pd.DataFrame(results)

,Model Name,Accuracy,Precision,Recall
0,Logistic Regression,0.897188,0.897086,0.897188


In [49]:
from sklearn.model_selection import RandomizedSearchCV


models = {
    "Logistic Regression": {
        "model": LogisticRegression(max_iter=1000),
        "params": {
            "C": [0.01, 0.1, 1, 10, 100],
            "solver": ["liblinear", "lbfgs"]
        }
    },
}

results = []

for name, config in models.items():

    random_search = RandomizedSearchCV(
        estimator=config["model"],
        param_distributions=config["params"],
        n_iter=10,
        cv=5,
        scoring="accuracy",
        random_state=42,
        n_jobs=-1
    )

    random_search.fit(X_train_BOW, y_train)

    best_model = random_search.best_estimator_

    y_predict = best_model.predict(X_test_BOW)

    results.append({
        "Model Name": name,
        "Best Parameters": random_search.best_params_,
        "Accuracy": accuracy_score(y_test, y_predict),
        "Precision": precision_score(y_test, y_predict, average="weighted"),
        "Recall": recall_score(y_test, y_predict, average="weighted")
    })

pd.DataFrame(results)

c:\Users\E L I T E\miniconda3\envs\ML\Lib\site-packages\sklearn\model_selection\_validation.py:490: FitFailedWarning: 
25 fits failed out of a total of 50.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
25 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\E L I T E\miniconda3\envs\ML\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\E L I T E\miniconda3\envs\ML\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\E L I T E\miniconda3\envs\ML\Lib\site-packages\sklearn\linear_m

,Model Name,Best Parameters,Accuracy,Precision,Recall
0,Logistic Regression,"{'solver': 'lbfgs', 'C': 10}",0.907188,0.906827,0.907188
